# =============================================================================
# OrthoCache Phase D: XLA Build Environment + Compaction Benchmark (Kaggle TPU)
# =============================================================================
# 
# This notebook sets up the XLA build environment on Kaggle's Linux TPU VM,
# builds a custom attention kernel that achieves TRUE dynamic loop elision,
# and benchmarks the wall-clock Δτ.
#
# Run each cell in order. Total setup time: ~15-30 minutes.
# =============================================================================

In [1]:
import jax
import jax.numpy as jnp
import sys, os, subprocess, shutil

print("=" * 60)
print("ENVIRONMENT DIAGNOSTICS")
print("=" * 60)
print(f"Python:     {sys.version}")
print(f"JAX:        {jax.__version__}")
print(f"Devices:    {jax.devices()}")
print(f"Device:     {jax.devices()[0].device_kind}")
print(f"Disk free:  {shutil.disk_usage('/').free / 1e9:.1f} GB")
print(f"gcc:        {subprocess.getoutput('gcc --version | head -1')}")
print(f"bazel:      {subprocess.getoutput('which bazel 2>/dev/null || echo NOT FOUND')}")
print("=" * 60)

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


ENVIRONMENT DIAGNOSTICS
Python:     3.12.13 (main, May 20 2026, 02:59:11) [GCC 14.2.0]
JAX:        0.10.1


E0000 00:00:1780407345.461461    3959 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


Devices:    [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]
Device:     TPU v5 lite
Disk free:  1179.1 GB
gcc:        gcc (Debian 14.2.0-19) 14.2.0
bazel:      NOT FOUND


In [2]:
# %% Cell 2: Clone OrthoCache + Install Dependencies
# ====================================================
import subprocess, os
# Clone the repo (skip if already cloned)
if not os.path.exists('/kaggle/working/orthocache'):
    subprocess.run(['git', 'clone', 'https://github.com/j-arndt/orthocache.git'], 
                   cwd='/kaggle/working', check=True)
else:
    subprocess.run(['git', 'pull', 'origin', 'main'], 
                   cwd='/kaggle/working/orthocache', check=True)
# Add to Python path
import sys
sys.path.insert(0, '/kaggle/working/orthocache/src')
print("OrthoCache source ready.")
print("Files in xla_extensions/:")
for f in os.listdir('/kaggle/working/orthocache/xla_extensions'):
    print(f"  {f}")

Already up to date.
OrthoCache source ready.
Files in xla_extensions/:
  orthocache_stream_compact.cc
  BUILD
  orthocache_stream_compact.h


From https://github.com/j-arndt/orthocache
 * branch            main       -> FETCH_HEAD


In [3]:
# %% Cell 3: Isolate the Bottleneck
# ===================================
# Before building XLA, let's figure out WHERE the 325ms is going.
# The Phase C benchmark compared dense (plain einsum) vs sparse/compact
# (full FWHT pipeline + Pallas kernel). That's not a fair comparison.
# 
# This cell benchmarks JUST the attention kernel with a pre-computed mask,
# separating pipeline overhead from actual MXU predication cost.
import time
import jax
import jax.numpy as jnp
from orthocache.sparse_attention import compile_pallas_sparse_attention, jax_block_sparse_attention
from orthocache.compaction import stream_compact
BLOCK_SIZE = 512
SEQ_LEN_K = 32768
NUM_HEADS = 16
HEAD_DIM = 256
NUM_BLOCKS = SEQ_LEN_K // BLOCK_SIZE
NUM_ITERS = 20
WARMUP = 3
key = jax.random.PRNGKey(42)
q = jax.random.normal(key, (1, NUM_HEADS, HEAD_DIM), dtype=jnp.bfloat16) / jnp.sqrt(HEAD_DIM)
keys = jax.random.normal(key, (SEQ_LEN_K, NUM_HEADS, HEAD_DIM), dtype=jnp.bfloat16)
values = jax.random.normal(key, (SEQ_LEN_K, NUM_HEADS, HEAD_DIM), dtype=jnp.bfloat16)
# Pre-compute masks with different eviction rates
def make_mask(eviction_pct):
    """Create a block mask with a given eviction percentage."""
    n_evict = int(NUM_BLOCKS * eviction_pct / 100)
    mask = jnp.ones((NUM_BLOCKS, NUM_HEADS), dtype=jnp.bool_)
    # Evict the last n_evict blocks
    if n_evict > 0:
        mask = mask.at[-n_evict:, :].set(False)
    return mask
def bench_kernel(label, fn, num_iters=NUM_ITERS, warmup=WARMUP):
    """Benchmark a function, returning average latency in ms."""
    # Warmup
    for _ in range(warmup):
        out = fn()
        out.block_until_ready()
    # Timed
    t0 = time.perf_counter()
    for _ in range(num_iters):
        out = fn()
        out.block_until_ready()
    t1 = time.perf_counter()
    avg_ms = (t1 - t0) / num_iters * 1000
    print(f"  {label}: {avg_ms:.3f} ms")
    return avg_ms
print("=" * 60)
print("ATTENTION KERNEL ISOLATION BENCHMARK")
print(f"Shape: Q=(1,{NUM_HEADS},{HEAD_DIM}), KV=({SEQ_LEN_K},{NUM_HEADS},{HEAD_DIM})")
print(f"Blocks: {NUM_BLOCKS}, Block size: {BLOCK_SIZE}")
print("=" * 60)
# 1. Dense baseline (raw einsum, no pipeline)
print("\n[1] Dense Attention (jnp.einsum):")
scale = jnp.sqrt(jnp.float32(HEAD_DIM))
dense_fn = lambda: jnp.einsum('qhd,khd->qkh', q, keys) / scale
dense_fn = jax.jit(dense_fn)
dense_ms = bench_kernel("dense_einsum", dense_fn)
# 2. Pallas sparse kernel with 0% eviction (all blocks retained)
print("\n[2] Pallas Sparse Kernel (0% eviction = full work):")
mask_0 = make_mask(0)
sparse_fn_0 = jax.jit(lambda: compile_pallas_sparse_attention(q, keys, values, mask_0, BLOCK_SIZE))
sparse_0_ms = bench_kernel("pallas_0pct", sparse_fn_0)
# 3. Pallas sparse kernel with 50% eviction
print("\n[3] Pallas Sparse Kernel (50% eviction):")
mask_50 = make_mask(50)
sparse_fn_50 = jax.jit(lambda: compile_pallas_sparse_attention(q, keys, values, mask_50, BLOCK_SIZE))
sparse_50_ms = bench_kernel("pallas_50pct", sparse_fn_50)
# 4. Pallas sparse kernel with 90% eviction
print("\n[4] Pallas Sparse Kernel (90% eviction):")
mask_90 = make_mask(90)
sparse_fn_90 = jax.jit(lambda: compile_pallas_sparse_attention(q, keys, values, mask_90, BLOCK_SIZE))
sparse_90_ms = bench_kernel("pallas_90pct", sparse_fn_90)
# 5. Pallas sparse kernel with 100% eviction (zero work — should be fast if no predication)
print("\n[5] Pallas Sparse Kernel (100% eviction = all masked):")
mask_100 = make_mask(100)
sparse_fn_100 = jax.jit(lambda: compile_pallas_sparse_attention(q, keys, values, mask_100, BLOCK_SIZE))
sparse_100_ms = bench_kernel("pallas_100pct", sparse_fn_100)
print("\n" + "=" * 60)
print("SUMMARY: PREDICATION PROOF")
print("=" * 60)
print(f"{'Eviction %':<15} | {'Latency (ms)':<15} | {'vs 0% eviction':<15}")
print("-" * 50)
for label, ms in [("0%", sparse_0_ms), ("50%", sparse_50_ms), 
                   ("90%", sparse_90_ms), ("100%", sparse_100_ms)]:
    ratio = ms / sparse_0_ms
    print(f"{label:<15} | {ms:<15.3f} | {ratio:<15.3f}x")
print(f"\nDense einsum:   {dense_ms:.3f} ms")
print(f"Pallas 0%:      {sparse_0_ms:.3f} ms")
print(f"Pallas 100%:    {sparse_100_ms:.3f} ms")
if sparse_0_ms > 0:
    speedup_at_100 = (sparse_0_ms - sparse_100_ms) / sparse_0_ms * 100
    print(f"\nSpeedup at 100% eviction: {speedup_at_100:.1f}%")
    if abs(speedup_at_100) < 5:
        print(">>> CONFIRMED: MXU predication. Zero eviction produces zero speedup.")
        print(">>> XLA executes all loop iterations regardless of mask.")
    else:
        print(">>> Partial speedup detected. Pallas may have some dynamic elision.")

ATTENTION KERNEL ISOLATION BENCHMARK
Shape: Q=(1,16,256), KV=(32768,16,256)
Blocks: 64, Block size: 512

[1] Dense Attention (jnp.einsum):
  dense_einsum: 0.645 ms

[2] Pallas Sparse Kernel (0% eviction = full work):
  pallas_0pct: 1.047 ms

[3] Pallas Sparse Kernel (50% eviction):
  pallas_50pct: 1.008 ms

[4] Pallas Sparse Kernel (90% eviction):
  pallas_90pct: 1.000 ms

[5] Pallas Sparse Kernel (100% eviction = all masked):
  pallas_100pct: 0.632 ms

SUMMARY: PREDICATION PROOF
Eviction %      | Latency (ms)    | vs 0% eviction 
--------------------------------------------------
0%              | 1.047           | 1.000          x
50%             | 1.008           | 0.963          x
90%             | 1.000           | 0.955          x
100%            | 0.632           | 0.603          x

Dense einsum:   0.645 ms
Pallas 0%:      1.047 ms
Pallas 100%:    0.632 ms

Speedup at 100% eviction: 39.7%
>>> Partial speedup detected. Pallas may have some dynamic elision.


In [1]:
# %% Cell 4 (REPLACEMENT): Build CustomCall with simple interface
import subprocess, os

WORKDIR = '/kaggle/working/phase_d_build'
os.makedirs(WORKDIR, exist_ok=True)

custom_call_src = r'''
#include <cstdint>
#include <cstring>
#include <cmath>
#include <algorithm>

extern "C" {

void orthocache_compacted_attention(
    float* out,
    const float* q,
    const float* keys,
    const float* values,
    const int32_t* block_mask,
    int32_t seq_len_q,
    int32_t head_dim,
    int32_t block_size,
    int32_t num_blocks
) {
    float scale = 1.0f / sqrtf((float)head_dim);
    
    // Stream compaction: build active index list
    int32_t active_indices[8192];
    int32_t num_active = 0;
    for (int32_t b = 0; b < num_blocks; ++b) {
        if (block_mask[b] != 0) {
            active_indices[num_active++] = b;
        }
    }
    
    // Online softmax over ONLY active blocks
    for (int32_t qi = 0; qi < seq_len_q; ++qi) {
        float r_max = -1e9f;
        float r_sum = 0.0f;
        float* r_out = (float*)calloc(head_dim, sizeof(float));
        
        for (int32_t i = 0; i < num_active; ++i) {
            int32_t b = active_indices[i];
            int32_t block_start = b * block_size;
            
            for (int32_t t = 0; t < block_size; ++t) {
                int32_t k_idx = block_start + t;
                
                float logit = 0.0f;
                for (int32_t d = 0; d < head_dim; ++d) {
                    logit += q[qi * head_dim + d] * keys[k_idx * head_dim + d];
                }
                logit *= scale;
                
                float new_max = std::max(r_max, logit);
                float exp_logit = expf(logit - new_max);
                float scale_old = expf(r_max - new_max);
                
                r_sum = r_sum * scale_old + exp_logit;
                for (int32_t d = 0; d < head_dim; ++d) {
                    r_out[d] = r_out[d] * scale_old + exp_logit * values[k_idx * head_dim + d];
                }
                r_max = new_max;
            }
        }
        
        float inv_sum = (r_sum > 1e-9f) ? (1.0f / r_sum) : 0.0f;
        for (int32_t d = 0; d < head_dim; ++d) {
            out[qi * head_dim + d] = r_out[d] * inv_sum;
        }
        free(r_out);
    }
}

}
'''

src_path = os.path.join(WORKDIR, 'orthocache_custom_call.cc')
with open(src_path, 'w') as f:
    f.write(custom_call_src)

so_path = os.path.join(WORKDIR, 'liborthocache_custom_call.so')
result = subprocess.run(
    ['g++', '-shared', '-fPIC', '-O3', '-march=native', '-o', so_path, src_path, '-lm'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"COMPILE ERROR:\n{result.stderr}")
else:
    print(f"SUCCESS: {so_path} ({os.path.getsize(so_path)/1024:.1f} KB)")

SUCCESS: /kaggle/working/phase_d_build/liborthocache_custom_call.so (15.2 KB)


In [2]:
# %% Cell 5: Benchmark dynamic loop
import ctypes, time
import numpy as np

so_path = '/kaggle/working/phase_d_build/liborthocache_custom_call.so'
lib = ctypes.CDLL(so_path)
fn = lib.orthocache_compacted_attention
fn.restype = None
fn.argtypes = [
    ctypes.c_void_p,  # out
    ctypes.c_void_p,  # q
    ctypes.c_void_p,  # keys
    ctypes.c_void_p,  # values
    ctypes.c_void_p,  # block_mask
    ctypes.c_int32,   # seq_len_q
    ctypes.c_int32,   # head_dim
    ctypes.c_int32,   # block_size
    ctypes.c_int32,   # num_blocks
]

SEQ_LEN_K = 32768
HEAD_DIM = 256
BLOCK_SIZE = 512
NUM_BLOCKS = SEQ_LEN_K // BLOCK_SIZE
NUM_ITERS = 20
WARMUP = 3

np.random.seed(42)
q_np = (np.random.randn(1, HEAD_DIM) / np.sqrt(HEAD_DIM)).astype(np.float32)
keys_np = np.random.randn(SEQ_LEN_K, HEAD_DIM).astype(np.float32)
values_np = np.random.randn(SEQ_LEN_K, HEAD_DIM).astype(np.float32)

def bench(eviction_pct):
    n_evict = int(NUM_BLOCKS * eviction_pct / 100)
    mask = np.ones(NUM_BLOCKS, dtype=np.int32)
    if n_evict > 0:
        mask[-n_evict:] = 0
    out = np.zeros((1, HEAD_DIM), dtype=np.float32)
    
    for _ in range(WARMUP):
        fn(out.ctypes.data, q_np.ctypes.data, keys_np.ctypes.data,
           values_np.ctypes.data, mask.ctypes.data, 1, HEAD_DIM, BLOCK_SIZE, NUM_BLOCKS)
    
    t0 = time.perf_counter()
    for _ in range(NUM_ITERS):
        fn(out.ctypes.data, q_np.ctypes.data, keys_np.ctypes.data,
           values_np.ctypes.data, mask.ctypes.data, 1, HEAD_DIM, BLOCK_SIZE, NUM_BLOCKS)
    t1 = time.perf_counter()
    return (t1 - t0) / NUM_ITERS * 1000

print("=" * 60)
print("PHASE D: DYNAMIC LOOP BENCHMARK (C++ on host CPU)")
print(f"Single head, Q=(1,{HEAD_DIM}), KV=({SEQ_LEN_K},{HEAD_DIM})")
print("=" * 60)

results = {}
for pct in [0, 25, 50, 75, 90, 100]:
    ms = bench(pct)
    results[pct] = ms
    print(f"  evict_{pct}%: {ms:.3f} ms")

print(f"\n{'Eviction %':<12} | {'Latency (ms)':<14} | {'Speedup':<10} | {'Δτ (ms)':<10}")
print("-" * 52)
base = results[0]
for pct in [0, 25, 50, 75, 90, 100]:
    ms = results[pct]
    print(f"{pct:<12} | {ms:<14.3f} | {base/ms:<9.2f}x | {base-ms:<10.3f}")

PHASE D: DYNAMIC LOOP BENCHMARK (C++ on host CPU)
Single head, Q=(1,256), KV=(32768,256)
  evict_0%: 8.395 ms
  evict_25%: 6.506 ms
  evict_50%: 4.311 ms
  evict_75%: 2.226 ms
  evict_90%: 0.910 ms
  evict_100%: 0.007 ms

Eviction %   | Latency (ms)   | Speedup    | Δτ (ms)   
----------------------------------------------------
0            | 8.395          | 1.00     x | 0.000     
25           | 6.506          | 1.29     x | 1.889     
50           | 4.311          | 1.95     x | 4.084     
75           | 2.226          | 3.77     x | 6.169     
90           | 0.910          | 9.22     x | 7.485     
100          | 0.007          | 1233.93  x | 8.388     
